In [8]:
import pandas as pd
from pathlib import Path
from google.cloud import bigquery
from datetime import timedelta, datetime

In [2]:
data_dir = Path.cwd().parent.parent / 'data' / 'olist_data'
files = sorted(data_dir.glob("*.csv"))

In [ ]:
trash = pd.read_csv(data_dir / 'olist_order_payments_dataset.csv')
trash.payment_installments.drop_duplicates().sort_values()

In [17]:
rev = pd.read_csv(data_dir / 'olist_order_reviews_dataset.csv')
rev[rev['review_id'].duplicated()]
rev[rev.review_id=='3242cc306a9218d0377831e175d62fbf']

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
1009,3242cc306a9218d0377831e175d62fbf,711246846695fa895c1776f0ddc30b20,5,NaN,Ótimo,2017-07-21 00:00:00,2017-07-22 01:06:37
3317,3242cc306a9218d0377831e175d62fbf,9c5bfba7de6a4abbb6ba0baab78d1622,5,NaN,Ótimo,2017-07-21 00:00:00,2017-07-22 01:06:37


In [22]:
prod = pd.read_csv(data_dir / 'olist_products_dataset.csv')

In [36]:
prod[(prod.product_weight_g % 1 != 0) & (prod.product_weight_g.notna())]

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm


In [31]:
prod.head()

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0


# Part 1.1: Exploring the data
* note, for customer data, key to orders data is customer_id

In [244]:
# lets look at order status first to see what orders we care aobut
df_raw_order = pd.read_csv(data_dir / 'olist_orders_dataset.csv')
df_order = df_raw_order[df_raw_order.order_status=='delivered']

df_customer = pd.read_csv(data_dir / 'olist_customers_dataset.csv')

df_sales_raw = pd.read_csv(data_dir / 'olist_order_payments_dataset.csv')
df_sales_agg = df_sales_raw.groupby('order_id')[['payment_value']].sum()

df_order_items = (pd.read_csv(data_dir / 'olist_order_items_dataset.csv', usecols=['order_id', 'order_item_id', 'product_id', 'seller_id', 'price', 'freight_value'])
                  .assign(total_price=lambda df_: df_.price + df_.freight_value)
                  )

df_products_raw = pd.read_csv(data_dir / 'olist_products_dataset.csv')
df_product_cat =  pd.read_csv(data_dir / 'product_category_name_translation.csv')

df_products = df_products_raw.merge(df_product_cat,left_on='product_category_name', right_on='product_category_name')[['product_id', 'product_category_name_english']]

df_order_reviews_raw = pd.read_csv(data_dir / 'olist_order_reviews_dataset.csv', usecols=['order_id', 'review_score'])

In [290]:
df_sellers_raw = pd.read_csv(data_dir / 'olist_sellers_dataset.csv')
df_sellers_raw

,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ
3,c0f3eea2e14555b6faeea3dd58c1b1c3,4195,sao paulo,SP
4,51a04a8a6bdcb23deccc82b0b80742cf,12914,braganca paulista,SP
...,...,...,...,...
3090,98dddbc4601dd4443ca174359b237166,87111,sarandi,PR
3091,f8201cab383e484733266d1906e2fdfa,88137,palhoca,SC
3092,74871d19219c7d518d0090283e03c137,4650,sao paulo,SP
3093,e603cf3fec55f8697c9059638d6c8eb5,96080,pelotas,RS


In [ ]:
df_order_items[df_order_items.order_item_id!=1]
df_order_items[df_order_items.order_id=='ca3625898fbd48669d50701aba51cd5f']
# df_order_items.groupby('order_id')['product_id'].nunique().sort_values()
# df_order_items

In [95]:
# now perform by pulling from google cloud instead
client = bigquery.Client()


table_id = 'msds-data-platform.raw.orders'
q = """
select order_id, customer_id, order_purchase_timestamp, order_delivered_customer_date
from raw.orders
where 1=1
    and order_status = @status
limit 5
"""
config = bigquery.QueryJobConfig(
    query_parameters=[bigquery.ScalarQueryParameter("status", 'STRING', 'delivered')]
)
df_order = client.query(q, job_config=config).to_dataframe()
client.close()

/Users/michaelcrapo/Documents/GitHub/personal/msds-data-platform/.venv/lib/python3.14/site-packages/google/cloud/bigquery/table.py:2131: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


In [40]:
df_customer[df_customer.customer_unique_id=='8d50f5eadf50201ccdcedfb9e2ac8455'].head(1)

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
14186,1bd3585471932167ab72a84955ebefea,8d50f5eadf50201ccdcedfb9e2ac8455,4045,sao paulo,SP


In [ ]:
df_customer.groupby('customer_unique_id')[['customer_id']].count().sort_values('customer_id')
df_sales_agg

* lets measure the repeat-purchasers.  find top 10% of customers by order #s and total spent

In [140]:
df_order_items.head(1)

,order_id,order_item_id,product_id,seller_id
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202


In [249]:
df = (df_order[['order_id', 'customer_id', 'order_purchase_timestamp', 'order_delivered_customer_date']]
 .merge(
    df_customer[['customer_id', 'customer_unique_id', 'customer_zip_code_prefix']],
    on='customer_id',
    how='left'
)
# .set_index('order_id')

.merge(
    df_sales_agg.reset_index(names="order_id"),
    on='order_id',
    how='left'
)
.merge(
    df_order_items,
    on='order_id',
    how='left'
)
.merge(
    df_products,
    on='product_id',
    how='left'
)
.assign(
    order_purchase_timestamp=lambda df_: pd.to_datetime(df_.order_purchase_timestamp, format="%Y-%m-%d %H:%M:%S",errors="coerce")
)
.merge(
    df_order_reviews_raw,
    on='order_id',
    how='left'
)
)
# df_customer[['customer_id', 'customer_unique_id', 'customer_zip_code_prefix']].dtypes

In [218]:
# by total orders
df_sales_pcts = (df
 .groupby(['customer_unique_id'])[['total_price']].sum().sort_values(by='total_price',ascending=False)
 .assign(pct_total_sales=lambda df_: df_['total_price']/df_['total_price'].sum())
 .assign(cum_pct=lambda df_: df_['pct_total_sales'].cumsum())
)

In [219]:
top_10_pct_rev = df_sales_pcts[df_sales_pcts.cum_pct<=.1]

top_10_pct_rev.shape[0] / df_sales_pcts.shape[0]

0.00947963752436856

In [ ]:
df.head(1)

In [ ]:
# repeat-purchase rate : total customer with 2+ orders / customers with single purchase
# drop duplicates bc join fanned out products per order
rep_purch_rate_df = df.drop_duplicates('order_id', keep='first').groupby(['customer_unique_id'])[['order_id']].count().sort_values('order_id')
single_purchasers = rep_purch_rate_df[rep_purch_rate_df.order_id==1].shape[0]
multi_purchasers = rep_purch_rate_df[rep_purch_rate_df.order_id>1].shape[0]
print(f"customer rep rate: {multi_purchasers/single_purchasers:.2f} | {multi_purchasers} multi | {single_purchasers} single")

customer rep rate: 0.03 | 2801 multi | 90557 single


In [ ]:
# repeat-purchase rate : total customer with 2+ orders / customers with single purchase
# now, considering total items ordered, not just single order
rep_purch_rate_df = df.groupby(['customer_unique_id'])[['order_id']].count().sort_values('order_id')
single_purchasers = rep_purch_rate_df[rep_purch_rate_df.order_id==1].shape[0]
multi_purchasers = rep_purch_rate_df[rep_purch_rate_df.order_id>1].shape[0]
print(f"customer rep rate: {multi_purchasers/single_purchasers:.2f} | {multi_purchasers} multi | {single_purchasers} single")

customer rep rate: 0.14 | 11610 multi | 81748 single


In [ ]:
# now lets find the latest date each product was purchased for each customer.  we will want to be able to say only count items before max date in dataset - t months
max_date = df.order_purchase_timestamp.max()
date_cutoffs = max_date - timedelta(days=0)
df[df.order_purchase_timestamp<=date_cutoffs].groupby(['customer_unique_id', 'product_id'])['order_purchase_timestamp'].max().rename('latest_purchase_datetime')

In [252]:
max_date = df.order_purchase_timestamp.max()
date_cutoffs = max_date - timedelta(days=0)
tds = df.join(
    df[df.order_purchase_timestamp<=date_cutoffs].groupby(['customer_unique_id', 'product_id'])['order_purchase_timestamp'].max().rename('latest_purchase_datetime'),
    on=["customer_unique_id", "product_id"]
)

In [ ]:
# find rpr given cutoff date
date_cutoffs = max_date - timedelta(days=30)
rep_purch_rate_df = tds.dropna().drop_duplicates('order_id', keep='first').groupby(['customer_unique_id'])[['order_id']].count().sort_values('order_id')
single_purchasers = rep_purch_rate_df[rep_purch_rate_df.order_id==1].shape[0]
multi_purchasers = rep_purch_rate_df[rep_purch_rate_df.order_id>1].shape[0]
print(f"customer rep rate: {multi_purchasers/single_purchasers:.2f} | {multi_purchasers} multi | {single_purchasers} single")

customer rep rate: 0.03 | 2742 multi | 89328 single


* now lets investigate if the classic 80/20 rule for each product plays out by # of orders and total rev

In [241]:
tds['days_since_last_purch'] = (tds['latest_purchase_datetime'] - tds['order_purchase_timestamp']).dt.days

In [309]:
# average review per product
reviews_agg = (tds
 .groupby(['product_category_name_english'])
 [['product_category_name_english', 'review_score']]
 .agg(
    avg_review_score=("review_score", "mean"),
    std_review_score=("review_score", "std"),
    orders=("product_category_name_english", "count"),
    )
.assign(avg_review_score=lambda df_: df_.avg_review_score.round(2))
 .sort_values(['avg_review_score'], ascending=True)
)

In [310]:
# need to find products each customer repurchased. then can divide the multi purchase by total # of purchase for product
rds = tds.groupby(['customer_unique_id', 'product_category_name_english'])[['total_price']].agg(num_orders=('total_price', 'count'), total_spent=('total_price', 'sum'))# .count()

In [311]:
repeat_df = rds[rds.num_orders>1].reset_index().groupby('product_category_name_english')[['num_orders', 'total_spent']].sum().sort_values('total_spent',ascending=False)

In [308]:
reviews_agg.avg_review_score.mean()

np.float64(4.102112676056338)

In [314]:
ragg = reviews_agg.join(repeat_df, how='right').assign(zscore=lambda df_: (df_.avg_review_score-df_.avg_review_score.mean())/df_.avg_review_score.std())

### real proof of the 80/20 rule!

In [282]:
tds[['total_price']].agg(num_orders=('total_price', 'count'), total_spent=('total_price', 'sum')).iloc[:,0]

num_orders       110840.00
total_spent    15489665.55
Name: total_price, dtype: float64

In [283]:
# proof that 20% of reorders comes from multi orders.  ~17% of total rev comes from repurchasers
rds[rds.num_orders>1].sum() / tds[['total_price']].agg(num_orders=('total_price', 'count'), total_spent=('total_price', 'sum')).iloc[:,0]

num_orders     0.225586
total_spent    0.169375
dtype: float64

In [ ]:
tds[tds.customer_unique_id=='fffcf5a5ff07b0908bd4e2dbc735a684']